In [ ]:
# Step 1: Import necessary libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This version uses relative file paths for local execution.


In [ ]:
#Prepare the required

# Base URL for the initial page with page parameter set to 1
base_url = "https://www.noon.com/saudi-en/fashion/men-31225/shoes-17421/sneakers-sportshoes-FA_03/?isCarouselView=false&limit=50&page=1&sort%5Bby%5D=popularity&sort%5Bdir%5D=desc"

# List to store product data
products_List = []

In [ ]:
# Step 2: Data Collection

# Loop to iterate over the first 34 pages
for page_number in range(1, 21):

    # Construct the URL for the current page
    page_url = f"https://www.noon.com/saudi-en/fashion/men-31225/shoes-17421/sneakers-sportshoes-FA_03/?isCarouselView=false&limit=50&page={page_number}&sort%5Bby%5D=popularity&sort%5Bdir%5D=desc"
    response = requests.get(page_url)
    soup = BeautifulSoup(response.content, "html.parser")
    grid_products = soup.find("div", class_="sc-63bc8e9b-7 hZVQUI grid")
    products = grid_products.find_all("span", class_="sc-980f39e6-0 iNoaVH wrapper productContainer")
    for product in products:
        # Extract product name
        name = product.find("div", class_="sc-95ea18ef-25 dOJDLL")
        name = name.text
        length = len(name)

        # Extract product price
        price = product.find("strong", class_="amount")
        price = price.text if price else "N/A"

        # Extract number of reviews
        num_reviews = product.find("span", class_="sc-9cb63f72-5 DkxLK")
        num_reviews = num_reviews.text if num_reviews else "N/A"

        # Extract product rating
        rating = product.find("div", class_="sc-9cb63f72-2 dGLdNc")
        rating = rating.text if rating else "N/A"

        # Extract original price(oldPrice)
        original_price = product.find("span", class_="oldPrice")
        original_price = original_price.text if original_price else "N/A"

        # Extract discount percentage
        discount = product.find("span", class_="discount")
        discount = discount.text if discount else "N/A"

        # Extract release date
        release_date = product.find("div", class_="sc-95ea18ef-34 gZsaNM")
        release_date = release_date.text if release_date else "N/A"

        # Append the extracted data to the products list
        products_List.append([name[:length-3], price, num_reviews, rating, original_price, discount[:2], release_date[10:]])

In [ ]:
# Create a DataFrame with the product data
df_product = pd.DataFrame(products_List, columns=[
    'Product Name', 'Price', 'Number of Reviews',
    'Rating', 'Original Price', 'Discount', 'Release Date'
    ])

In [ ]:
# Export data to CSV file before processing

# Export data to CSV file in Google Drive
df_product.to_csv('products_data_from_noon.csv', index=False)

# Export data to CSV file in my dvice
df_product.to_csv('products_data_from_noon.csv', index=False)

In [ ]:
# Step 3: Data Exploration
print(df_product.head())  # Show first 5 records
print('*'*100)
print(df_product.info())  # Display basic information about the dataset
print('*'*100)
print(df_product.describe())  # Summary statistics for numerical features

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_product.columns)

In [ ]:
# Step 4: Data Cleaning

# Check for missing values
print("Missing values in the dataset:")
print(df_product.isnull().sum()) # Find out which columns contain empty values

# Display the unique categories for text columns
print("Unique Product Names:", len(df_product['Product Name'].unique()))
print("Unique Categories:", df_product['Discount'].unique())

In [ ]:
# Handle missing values

# Replace "N/A" with NaN
df_product['Price'] = df_product['Price'].replace("N/A", np.nan)
# Convert values to string before replacing commas
df_product['Price'] = df_product['Price'].astype(str).str.replace(",", "", regex=True)
# Convert values to float
df_product['Price'] = df_product['Price'].astype(float)
# Fill missing values with the mean
df_product['Price'].fillna(df_product['Price'].mean(), inplace=True)


# Fill missing ratings with the median rating
df_product['Rating'] = df_product['Rating'].replace("N/A", np.nan).astype(float)
df_product['Rating'].fillna(df_product['Rating'].median(), inplace=True)

# Convert 'Number of Reviews' to numeric, forcing errors to NaN
df_product['Number of Reviews'] = pd.to_numeric(df_product['Number of Reviews'], errors='coerce')
# Fill missing values with the median
df_product['Number of Reviews'].fillna(df_product['Number of Reviews'].median(), inplace=True)


# Convert 'Original Price' to numeric, forcing errors to NaN
df_product['Original Price'] = pd.to_numeric(df_product['Original Price'], errors='coerce')
# Fill missing values with the mean
df_product['Original Price'].fillna(df_product['Original Price'].mean(), inplace=True)

# Convert 'Discount' to numeric, forcing errors to NaN
df_product['Discount'] = pd.to_numeric(df_product['Discount'], errors='coerce')
# Fill missing values with the median
df_product['Discount'].fillna(df_product['Discount'].median(), inplace=True)

print(df_product.isnull().sum())

In [ ]:
# Check for duplicated values
print("duplicated values in the dataset:")
print(df_product.duplicated().sum())

In [ ]:
# Handle duplicated values
df_product.drop_duplicates(inplace=True)
#data.drop_duplicates()

print(df_product.duplicated().sum())# Ensure that there are no empty values ​​after processing.

In [ ]:
# Convert discount percentage to numeric
#I did this step using slicing when I collected the data(but this is another solution too)
#df_product['Discount Percentage'] = df_product['Discount Percentage'].str.replace('%', '').astype(float)

# Check the cleaned data
df_product.info()

In [ ]:
# Step 5: Data Transformation

from sklearn.preprocessing import MinMaxScaler

# Normalize Price and Rating
scaler = MinMaxScaler()
df_product[['Price', 'Rating']] = scaler.fit_transform(df_product[['Price', 'Rating']])


# Apply pd.cut to categorize discounts
df_product['Discount Category'] = pd.cut(
    df_product['Discount'],
    bins=[-1, 0, 25, 50, 75, 100],
    labels=["No Discount", "Low", "Medium", "High", "Very High"]
)

In [ ]:
# Display data after conversions
print(df_product.head())

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_product.columns)

In [ ]:
df_product.info()
print('='*100)
print(df_product.isnull().sum())
print('='*100)

In [ ]:
# Step 6: Feature Engineering

# Create new feature: Product Name Length (number of characters in Product Name)
df_product['Product Name Length'] = df_product['Product Name'].apply(len)

# Create a new feature: Price-to-Rating Ratio
df_product['Price/Rating Ratio'] = df_product['Price'] / (df_product['Rating'] + 1e-5)

# Preview new features
df_product[['Product Name', 'Price/Rating Ratio']].head()

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_product.columns)
print('='*100)
df_product.info()
print('='*100)
print(df_product.isnull().sum())
print('='*100)

In [ ]:
# Step 7: visualizations

# Plot the distribution of product prices
plt.figure(figsize=(10, 6))
sns.histplot(df_product['Price'], kde=True, bins=30, color='blue')
plt.title("Price Distribution", fontsize=16)
plt.xlabel("Price", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.show()

In [ ]:
# Plot the distribution of product ratings
plt.figure(figsize=(10, 6))
sns.histplot(df_product['Rating'], kde=True, bins=10, color='orange')
plt.title("Rating Distribution", fontsize=16)
plt.xlabel("Rating", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.show()

In [ ]:
# Additional step => Data visualizations
df_product.hist()

In [ ]:
# Create a boxplot to distribute ratings
df_product.boxplot()

In [ ]:
# Step 8: Summary
print("""
Steps Completed:
1. Data Collection: Scraped product data and stored it in a structured format.
2. Data Exploration: Identified missing values, data types, and unique categories.
3. Data Cleaning: Handled missing values, duplicates, and inconsistent formats.
4. Data Transformation: Scaled numeric data and encoded categorical variables.
5. Feature Engineering: Created new features for analysis, such as 'Price/Rating Ratio' and 'Is New Product'.
""")

# Display the final transformed dataset
#df_posts.head()
print('='*80)
print("Programing by: Mahmoud M. M. Ashour")
print("*****Assignment 1: Data Collection and Preparation(Dataset 2 with Scraping)*****")
print("*****Advanced Data Science*****")
print("*****Dr. Iyad Husni Alshami*****")
print('='*80)

In [ ]:
# Export data to CSV file after processing

# Export data to CSV file in Google Drive
df_product.to_csv('processed_products_data_from_noon.csv', index=False)

# Export data to CSV file in my dvice
df_product.to_csv('processed_products_data_from_noon.csv', index=False)
